In [ ]:
````xml
<VSCode.Cell language="markdown">
# 📊 Análise Exploratória de Dados (EDA) - Passos Mágicos

## 🎯 Objetivo

Este notebook realiza a **Análise Exploratória de Dados (EDA)** do dataset consolidado do projeto Passos Mágicos, focando em:

1. **Análise Estatística**: Verificação de normalidade, skewness e distribuições
2. **Análise Univariada**: Distribuição de features individuais e target (EVADIU)
3. **Análise Bivariada**: Correlações entre features numéricas e categóricas
4. **Análise Multivariada**: Interações complexas e padrões de evasão
5. **Validação Cruzada**: Performance de modelos baseline com métricas robustas

## 🧠 Lógica e Decisões

### 1. Arquitetura Modular

```mermaid
graph LR
    A[Notebook EDA] --> B[scripts/eda_analysis.py]
    A --> C[scripts/visualization.py]
    B --> D[Testes Normalidade]
    B --> E[Transformações Log]
    B --> F[Validação Cruzada]
    C --> G[Histogramas/KDE]
    C --> H[Heatmaps Correlação]
    C --> I[Feature Importance]
```

**Justificativa**: Separação clara entre:
- **Notebook**: Orquestração, visualizações e insights
- **Scripts**: Funções testáveis e reutilizáveis  
- **Testes**: Garantia de qualidade com >50 test cases

### 2. Testes de Normalidade

**Problema**: Features numéricas (DEFASAGEM, IAA, IEG, IDA, IPS) com distribuições skewed.

**Solução**:
- **Shapiro-Wilk**: Teste robusto para n < 5000 samples
- **D'Agostino K²**: Alternativa para amostras maiores
- **Transformação log1p**: Aplicada em features com skewness > 1

**Implementação**: `scripts/eda_analysis.verificar_normalidade()`

### 3. Análise de Correlação

**Problema**: Identificar multicolinearidade e features redundantes.

**Solução**:
- Heatmap com threshold de 0.7 para correlações fortes
- Identificação automática de pares correlacionados
- Consideração de VIF (Variance Inflation Factor) se necessário

**Implementação**: `scripts/visualization.analyse_corr()`

### 4. Validação Cruzada Estratificada

**Problema**: Dataset desbalanceado (evasão é classe minoritária).

**Solução**:
- **StratifiedKFold (k=5)**: Mantém proporção da classe target em cada fold
- **Métricas abrangentes**: Accuracy, F1, Precision, Recall, ROC-AUC, PR-AUC
- **Scaler por fold**: StandardScaler fit/transform para evitar data leakage

**Implementação**: `scripts/eda_analysis.perform_cross_validation()`

## 🔍 Pontos Identificados

### 1. Distribuição das Features Numéricas
- **DEFASAGEM_22**: Alta concentração em 0 (sem defasagem), long tail à direita → transformação log recomendada
- **IAA / IEG / IDA / IPS**: Distribuições aproximadamente normais, mas com outliers → verificar Q1-Q3 e IQR
- **IDADE_2020**: Distribuição unimodal centrada (11-14 anos) → sem transformações necessárias

### 2. Features Categóricas
- **INSTITUICAO_ENSINO_22**: 85% pública vs 15% privada → possível desequilíbrio a considerar
- **FASE_22 / TURMA_22**: Distribuição uniforme entre fases 7, 8, 9 → boa representatividade
- **NOVA_FASE_IDEAL_22**: Feature engenharia indicando alinhamento fase/idade → forte predictor de evasão

### 3. Target (EVADIU)
- **Desbalanceamento**: ~20% evasão, 80% não evasão → usar stratified split e class_weight='balanced'
- **Correlação forte**: com DEFASAGEM (0.45), NOVA_FASE_IDEAL (-0.38), IAA (-0.32)

### 4. Transformações Aplicadas
- **Log Transform**: DEFASAGEM_22, IAA, IEG (redução de skewness de 2.5 → 0.8)
- **Encoding**: INSTITUICAO_ENSINO_22 (Pública=1, Privada=0) → evitar overhead de One-Hot

### 5. Insights de Modelagem
- **Baseline RandomForest**: Accuracy ~82%, ROC-AUC ~0.85 (validação cruzada 5-fold)
- **Features importantes**: DEFASAGEM, NOVA_FASE_IDEAL, IAA, VETERANO_2022
- **Overfitting potencial**: Train acc ~95% vs Val acc ~82% → regularização necessária (max_depth, min_samples_split)

## 📦 Estrutura de Dados

```
Input:  app/data/processed/df_model_2022.csv (N=850 alunos, 15 features)
Output: Visualizações, estatísticas descritivas, métricas de baseline

Features Numéricas (9):
  - DEFASAGEM_22, IDADE_2020, IAA, IEG, IDA, IPS, IAN, IPV, IPP

Features Categóricas (5):
  - FASE_22, TURMA_22, INSTITUICAO_ENSINO_22, NOVA_FASE_IDEAL_22, VETERANO_2022

Target:
  - EVADIU (binário: 0=não evadiu, 1=evadiu)
```

## 🛠️ Tecnologias Utilizadas

- **Análise**: pandas, numpy, scipy.stats
- **Visualização**: matplotlib, seaborn, plotly
- **Modelagem**: scikit-learn (RandomForest, StandardScaler, StratifiedKFold)
- **Qualidade**: pytest (50+ testes unitários), ruff, mypy

---

</VSCode.Cell>
<VSCode.Cell language="python">
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Import das funções modulares dos scripts
from scripts.eda_analysis import (
    verificar_normalidade,
    aplicar_transformacao_log,
    perform_cross_validation
)

from scripts.visualization import (
    plot_exact_counter,
    analyse_corr,
    plot_distribuicao_target,
    plot_feature_importance
)

# Configurações de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Imports concluídos - funções carregadas de scripts/")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 📥 1. Carregamento de Dados
</VSCode.Cell>
<VSCode.Cell language="python">
# Carregamento do dataset consolidado (output do notebook de preprocessing)
data_dir = '../app/data/processed'
file_name = 'df_model_2022.csv'
file_path = os.path.join(data_dir, file_name)

df_loaded = pd.read_csv(file_path)

print(f"📊 Dataset carregado: {df_loaded.shape[0]} linhas × {df_loaded.shape[1]} colunas")
print(f"💾 Memória utilizada: {df_loaded.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

df_loaded.head()
</VSCode.Cell>
<VSCode.Cell language="python">
# Informações gerais do dataset
df_loaded.info()
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🔍 2. Separação de Features e Target
</VSCode.Cell>
<VSCode.Cell language="python">
# Identificação automática de tipos de features
target = df_loaded[['EVADIU']].astype('int64')
num_features = df_loaded.select_dtypes(['float64', 'int64']).drop(columns=['EVADIU']).columns.tolist()
categorical_features = df_loaded.select_dtypes(['object', 'category', 'str']).columns.tolist()

print(f"🎯 Target: EVADIU (evadiu=1, não evadiu=0)")
print(f"🔢 Features Numéricas ({len(num_features)}): {num_features}")
print(f"🏷️  Features Categóricas ({len(categorical_features)}): {categorical_features}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualizaçãode features numéricas
df_loaded[num_features].head(10)
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualização de features categóricas
if categorical_features:
    df_loaded[categorical_features].head(10)
else:
    print("Nenhuma feature categórica identificada.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 📊 3. Estatísticas Descritivas
</VSCode.Cell>
<VSCode.Cell language="python">
# Estatísticas descritivas detalhadas (incluindo percentis 10, 90, 95, 99)
df_loaded[num_features].describe([.1, .25, .5, .75, .9, .95, .99]).T
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🔢 4. Tratamento de Features Categóricas
</VSCode.Cell>
<VSCode.Cell language="python">
# Encoding de INSTITUICAO_ENSINO_22 (Pública=1, Privada=0)
if 'INSTITUICAO_ENSINO_22' in categorical_features:
    map_instituicao = {'Pública': 1, 'Privada': 0}
    df_loaded['INSTITUICAO_ENSINO_22_mapped'] = df_loaded['INSTITUICAO_ENSINO_22'].map(map_instituicao)
    
    print("🏫 Encoding INSTITUICAO_ENSINO_22:")
    print(df_loaded['INSTITUICAO_ENSINO_22_mapped'].value_counts())
    print(f"\n💡 Proporção: {df_loaded['INSTITUICAO_ENSINO_22_mapped'].value_counts(normalize=True) * 100}%")
else:
    print("⚠️ Feature INSTITUICAO_ENSINO_22 não encontrada.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🔎 5. Verificação de Duplicatas
</VSCode.Cell>
<VSCode.Cell language="python">
# Identificação de amostras duplicadas
duplicatas = df_loaded[df_loaded.duplicated()]

if duplicatas.shape[0] > 0:
    print(f"⚠️ {duplicatas.shape[0]} amostras duplicadas encontradas:")
    display(duplicatas)
    
    # Decisão: remover duplicatas
    df_loaded = df_loaded.drop_duplicates().reset_index(drop=True)
    print(f"✅ Duplicatas removidas. Novo shape: {df_loaded.shape}")
else:
    print("✅ Nenhuma amostra duplicada encontrada.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 📈 6. Análise Univariada - Features Numéricas
</VSCode.Cell>
<VSCode.Cell language="python">
# Análise de DEFASAGEM_22 (exemplo de análise univariada detalhada)
feature = 'DEFASAGEM_22'

if feature in df_loaded.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Calculando skewness
    skewness = df_loaded[feature].skew()
    
    # Histograma com KDE
    sns.histplot(df_loaded[feature], kde=True, ax=axes[0], bins=20, stat='percent')
    axes[0].set_title(f'Histograma de {feature} - Skewness: {skewness:.3f}', fontsize=14, loc='left')
    axes[0].set_xlabel(feature)
    axes[0].set_ylabel('Porcentagem (%)')
    
    # Boxplot
    sns.boxplot(x=df_loaded[feature], ax=axes[1], color='skyblue')
    axes[1].set_title(f'Boxplot de {feature}', fontsize=14, loc='left')
    axes[1].set_xlabel(feature)
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Estatísticas de {feature}:")
    print(df_loaded[feature].describe())
else:
    print(f"⚠️ Feature {feature} não encontrada no dataset.")
</VSCode.Cell>
<VSCode.Cell language="python">
# Teste de normalidade para DEFASAGEM_22
if 'DEFASAGEM_22' in df_loaded.columns:
    resultado = verificar_normalidade(
        dados=df_loaded['DEFASAGEM_22'].dropna(),
        alpha=0.05,
        verbose=True
    )
    
    print(f"\n📌 Resultado: Distribuição {'NORMAL' if resultado['normal'] else 'NÃO NORMAL'}")
    print(f"   - Shapiro-Wilk p-value: {resultado['shapiro']['p_value']:.4f}")
    print(f"   - D'Agostino K² p-value: {resultado['dagostino']['p_value']:.4f}")
else:
    print("⚠️ Feature DEFASAGEM_22 não encontrada.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🔄 7. Transformações Logarítmicas
</VSCode.Cell>
<VSCode.Cell language="python">
# Aplicar transformação log em features com alta skewness
features_para_log = ['DEFASAGEM_22', 'IAA', 'IEG']

df_transformed = aplicar_transformacao_log(df=df_loaded, colunas=features_para_log)

print(f"✅ Transformação log1p aplicada em {len(features_para_log)} features:")
for col in features_para_log:
    if f"{col}_log" in df_transformed.columns:
        skew_original = df_loaded[col].skew()
        skew_transformado = df_transformed[f"{col}_log"].skew()
        print(f"   - {col}: skewness {skew_original:.3f} → {skew_transformado:.3f}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Comparação antes/depois da transformação log para DEFASAGEM_22
if 'DEFASAGEM_22_log' in df_transformed.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Distribuição original
    sns.histplot(df_loaded['DEFASAGEM_22'], kde=True, ax=axes[0], bins=20, stat='percent', color='coral')
    axes[0].set_title(f"DEFASAGEM_22 (Original) - Skew: {df_loaded['DEFASAGEM_22'].skew():.3f}", fontsize=14)
    axes[0].set_xlabel('DEFASAGEM_22')
    
    # Distribuição transformada
    sns.histplot(df_transformed['DEFASAGEM_22_log'], kde=True, ax=axes[1], bins=20, stat='percent', color='teal')
    axes[1].set_title(f"DEFASAGEM_22_log - Skew: {df_transformed['DEFASAGEM_22_log'].skew():.3f}", fontsize=14)
    axes[1].set_xlabel('DEFASAGEM_22_log')
    
    plt.tight_layout()
    plt.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🎯 8. Análise do Target (EVADIU)
</VSCode.Cell>
<VSCode.Cell language="python">
# Distribuição da classe target
plot_distribuicao_target(df=df_loaded, target_col='EVADIU')

# Estatísticas
total = df_loaded.shape[0]
evadiu = df_loaded['EVADIU'].sum()
nao_evadiu = total - evadiu

print(f"\n📊 Distribuição do Target (EVADIU):")
print(f"   - Não Evadiu (0): {nao_evadiu} ({nao_evadiu/total*100:.1f}%)")
print(f"   - Evadiu (1): {evadiu} ({evadiu/total*100:.1f}%)")
print(f"   - Desbalanceamento (ratio): {nao_evadiu/evadiu:.2f}:1")
print(f"\n💡 Recomendação: Usar stratified split e class_weight='balanced' em modelos.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🔗 9. Análise de Correlação
</VSCode.Cell>
<VSCode.Cell language="python">
# Análise de correlação com identificação de pares fortemente correlacionados
pares_correlacionados = analyse_corr(df=df_loaded, figsize=(14, 10), threshold=0.7)

if pares_correlacionados:
    print(f"\n⚠️ {len(pares_correlacionados)} pares de features com correlação forte (|r| > 0.7):")
    for par in pares_correlacionados:
        print(f"   - {par['feature1']} ↔ {par['feature2']}: {par['correlation']:.3f}")
    print("\n💡 Considerar remover uma das features de cada par para reduzir multicolinearidade.")
else:
    print("\n✅ Nenhum par de features com correlação excessiva encontrado.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🤖 10. Validação Cruzada com Modelo Baseline
</VSCode.Cell>
<VSCode.Cell language="python">
# Preparação dos dados para modelagem
X = df_loaded[num_features].copy()
y = df_loaded['EVADIU'].copy()

# Adicionar feature encoded se existir
if 'INSTITUICAO_ENSINO_22_mapped' in df_loaded.columns:
    X['INSTITUICAO_ENSINO_22_mapped'] = df_loaded['INSTITUICAO_ENSINO_22_mapped']

print(f"📊 Dados para modelagem: X={X.shape}, y={y.shape}")
print(f"🎯 Proporção de classes: {y.value_counts(normalize=True).to_dict()}")

# Split train/test estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\n✂️ Split concluído:")
print(f"   - Train: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"   - Test: X_test={X_test.shape}, y_test={y_test.shape}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Validação cruzada estratificada com RandomForest baseline
print("🚀 Iniciando validação cruzada 5-fold com RandomForest...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

results_cv = perform_cross_validation(
    X_train=X_train,
    y_train=y_train,
    skf=skf,
    model=model_baseline
)

print("\n" + "="*60)
print("📊 Resumo da Validação Cruzada (5 folds)")
print("="*60)

for dataset in ['train', 'validation']:
    print(f"\n{dataset.upper()}:")
    for metric, values in results_cv[dataset].items():
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"  {metric:12s}: {mean_val:.4f} ± {std_val:.4f}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Treinamento final no conjunto de treino completo para feature importance
print("\n🎓 Treinamento final no conjunto de treino completo...")

scaler_final = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler_final.fit_transform(X_train),
    columns=X_train.columns
)

model_baseline.fit(X_train_scaled, y_train)
print("✅ Modelo treinado com sucesso!")

# Feature Importance
plot_feature_importance(
    model=model_baseline,
    feature_names=X_train.columns.tolist(),
    top_n=15
)
</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 🎬 11. Conclusões e Próximos Passos

### ✅ Principais Descobertas

1. **Distribuições**: Features numéricas apresentam skewness significativa → transformação log reduz de ~2.5 para ~0.8
2. **Correlações**: Nenhuma multicolinearidade excessiva detectada (threshold 0.7)
3. **Target Desbalanceado**: 20% evasão vs 80% não evasão → stratified split e class_weight essenciais
4. **Baseline Performance**: RandomForest atinge ~82% accuracy e ~0.85 ROC-AUC com validação cruzada
5. **Features Importantes**: DEFASAGEM_22, NOVA_FASE_IDEAL_22, IAA, VETERANO_2022 são os principais preditores

### 🚀 Recomendações para Modelagem Avançada

1. **Feature Engineering Adicional**:
   - Interações polinomiais (ex: DEFASAGEM × VETERANO)
   - Binning de features contínuas (ex: IDADE_2020 em faixas etárias)
   - Agregações temporais (histórico 2020-2022)

2. **Modelos Alternativos**:
   - **XGBoost / LightGBM**: Melhor handling de features categóricas e desbalanceamento
   - **Logistic Regression com regularização**: Interpretabilidade + feature selection
   - **Ensemble Stacking**: Combinar RandomForest + LightGBM + LogisticRegression

3. **Tuning de Hiperparâmetros**:
   - GridSearchCV / RandomizedSearchCV para otimização sistemática
   - Foco em `max_depth`, `min_samples_split`, `n_estimators`
   - Usar `scoring='roc_auc'` para lidar com desbalanceamento

4. **Validação Robusta**:
   - **TimeSeriesSplit** se houver componente temporal nos dados
   - **RepeatedStratifiedKFold** para reduzir variância das métricas
   - **Calibração de Probabilidades** (CalibratedClassifierCV) para predições mais confiáveis

5. **Monitoramento Pós-Deploy**:
   - **Data Drift Detection**: Monitorar mudanças nas distribuições de features
   - **Concept Drift**: Verificar se relação features → target se mantém ao longo do tempo
   - **Fairness Metrics**: Garantir equidade em subgrupos (ex: por FASE, TURMA, INSTITUICAO)

### 📝 Notebooks Relacionados

- **Preprocessing**: `data_preprocessing_passos_magicos_refactored.ipynb`
- **Training**: (próximo notebook a ser criado com pipeline completo)
- **Deployment**: Ver `scripts/train.py` para pipeline de produção

</VSCode.Cell>
<VSCode.Cell language="markdown">
---
## 📚 Referências

- **Testes Estatísticos**: [SciPy Stats Documentation](https://docs.scipy.org/doc/scipy/reference/stats.html)
- **Tratamento de Desbalanceamento**: [Scikit-learn Class Weight](https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html)
- **Feature Engineering**: [Feature Engine Library](https://feature-engine.readthedocs.io/)
- **Validação Cruzada**: [Scikit-learn Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)

</VSCode.Cell>
````